# Logistic Regression - All Features (Minimal Column Removal)

This notebook runs logistic regression using **all features** from the prepared dataset.
Data cleaning (column removal, missing value handling) was done in `data_preparation.ipynb`
and saved to `data/processed/partially_selected_features.csv`.

**Data source:** `data/processed/partially_selected_features.csv` (33 features + target)

**Remaining features: 33** (vs 20 in the MRMR version)

In [ ]:
import sys, os
sys.path.insert(0, os.path.join(os.getcwd(), '..'))

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.model_selection import train_test_split, cross_val_score, StratifiedKFold
from sklearn.metrics import (
    confusion_matrix, classification_report, roc_curve, auc,
    precision_recall_curve, f1_score, accuracy_score, precision_score,
    recall_score, roc_auc_score
)
import warnings
warnings.filterwarnings('ignore')

print('Imports loaded.')

## 1. Load Prepared Data

In [ ]:
df = pd.read_csv('../data/processed/partially_selected_features.csv')
print(f"Shape: {df.shape}")
print(f"Columns ({len(df.columns)}): {list(df.columns)}")
print(f"\nMissing values: {df.isnull().sum().sum()}")

## 3. Encode & Split

In [ ]:
# Target
y = (df['fraud_reported'] == 'Y').astype(int)
X = df.drop(columns=['fraud_reported'])

# Label encode categoricals
categorical_cols = X.select_dtypes(include=['object', 'str']).columns.tolist()
numerical_cols = X.select_dtypes(include=['int64', 'float64']).columns.tolist()
print(f"Categorical features ({len(categorical_cols)}): {categorical_cols}")
print(f"Numerical features ({len(numerical_cols)}): {numerical_cols}")

for col in categorical_cols:
    X[col] = LabelEncoder().fit_transform(X[col].astype(str))

# Split: 70/15/15 stratified (same as MRMR notebook)
X_temp, X_test, y_temp, y_test = train_test_split(X, y, test_size=0.15, random_state=42, stratify=y)
X_train, X_val, y_train, y_val = train_test_split(X_temp, y_temp, test_size=0.15/(1-0.15), random_state=42, stratify=y_temp)

print(f"\nTrain: {len(X_train)} | Val: {len(X_val)} | Test: {len(X_test)}")
print(f"Fraud rates — Train: {y_train.mean()*100:.1f}% | Val: {y_val.mean()*100:.1f}% | Test: {y_test.mean()*100:.1f}%")

# Scale
scaler = StandardScaler()
X_train_s = pd.DataFrame(scaler.fit_transform(X_train), columns=X_train.columns, index=X_train.index)
X_val_s = pd.DataFrame(scaler.transform(X_val), columns=X_val.columns, index=X_val.index)
X_test_s = pd.DataFrame(scaler.transform(X_test), columns=X_test.columns, index=X_test.index)

---
## Run 1: Baseline — All 33 Features (C=1.0, balanced)

In [ ]:
results_log = []

model_v1 = LogisticRegression(C=1.0, l1_ratio=0, solver='lbfgs', max_iter=1000,
                               class_weight='balanced', random_state=42)
model_v1.fit(X_train_s, y_train)

for name, Xs, ys in [('Train', X_train_s, y_train), ('Validation', X_val_s, y_val)]:
    y_pred = model_v1.predict(Xs)
    y_prob = model_v1.predict_proba(Xs)[:, 1]
    print(f"[{name}] Accuracy: {accuracy_score(ys, y_pred):.4f} | "
          f"Precision: {precision_score(ys, y_pred):.4f} | "
          f"Recall: {recall_score(ys, y_pred):.4f} | "
          f"F1: {f1_score(ys, y_pred):.4f} | "
          f"ROC AUC: {roc_auc_score(ys, y_prob):.4f}")

val_pred = model_v1.predict(X_val_s)
val_prob = model_v1.predict_proba(X_val_s)[:, 1]

results_log.append({
    'Run': 'V1: L2, C=1.0',
    'Val F1': f1_score(y_val, val_pred),
    'Val Precision': precision_score(y_val, val_pred),
    'Val Recall': recall_score(y_val, val_pred),
    'Val ROC AUC': roc_auc_score(y_val, val_prob),
})

---
## Run 2: Cross-Validation Stability

In [ ]:
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

for metric in ['f1', 'precision', 'recall', 'roc_auc']:
    scores = cross_val_score(model_v1, X_train_s, y_train, cv=cv, scoring=metric)
    print(f"{metric:<12}: {scores.mean():.4f} (+/- {scores.std():.4f})  | folds: {[f'{s:.3f}' for s in scores]}")

---
## Run 3: C Parameter Tuning

In [ ]:
C_values = [0.001, 0.01, 0.1, 0.5, 1.0, 5.0, 10.0, 50.0, 100.0]
c_results = []

for C in C_values:
    m = LogisticRegression(C=C, l1_ratio=0, solver='lbfgs', max_iter=1000,
                           class_weight='balanced', random_state=42)
    m.fit(X_train_s, y_train)
    vp = m.predict(X_val_s)
    vprob = m.predict_proba(X_val_s)[:, 1]
    tp = m.predict(X_train_s)
    c_results.append({
        'C': C,
        'Train F1': f1_score(y_train, tp),
        'Val F1': f1_score(y_val, vp),
        'Val Precision': precision_score(y_val, vp),
        'Val Recall': recall_score(y_val, vp),
        'Val ROC AUC': roc_auc_score(y_val, vprob),
    })

c_df = pd.DataFrame(c_results)
print(c_df.to_string(index=False))

best_c = c_df.loc[c_df['Val F1'].idxmax(), 'C']
print(f"\nBest C by Val F1: {best_c}")

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].semilogx(c_df['C'], c_df['Train F1'], 'o-', label='Train F1')
axes[0].semilogx(c_df['C'], c_df['Val F1'], 's-', label='Val F1')
axes[0].axvline(x=best_c, color='red', linestyle='--', alpha=0.5, label=f'Best C={best_c}')
axes[0].set_xlabel('C'); axes[0].set_ylabel('F1 Score')
axes[0].set_title('F1 vs Regularization'); axes[0].legend(); axes[0].grid(True, alpha=0.3)

axes[1].semilogx(c_df['C'], c_df['Val Precision'], 'o-', label='Precision')
axes[1].semilogx(c_df['C'], c_df['Val Recall'], 's-', label='Recall')
axes[1].axvline(x=best_c, color='red', linestyle='--', alpha=0.5, label=f'Best C={best_c}')
axes[1].set_xlabel('C'); axes[1].set_ylabel('Score')
axes[1].set_title('Precision/Recall vs Regularization'); axes[1].legend(); axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

---
## Run 4: Best C — Full Evaluation

In [ ]:
model_v2 = LogisticRegression(C=best_c, l1_ratio=0, solver='lbfgs', max_iter=1000,
                               class_weight='balanced', random_state=42)

cv_scores = cross_val_score(model_v2, X_train_s, y_train, cv=cv, scoring='f1')
print(f"CV F1 with C={best_c}: {cv_scores.mean():.4f} (+/- {cv_scores.std():.4f})")

model_v2.fit(X_train_s, y_train)

for name, Xs, ys in [('Train', X_train_s, y_train), ('Validation', X_val_s, y_val)]:
    y_pred = model_v2.predict(Xs)
    y_prob = model_v2.predict_proba(Xs)[:, 1]
    print(f"[{name}] Accuracy: {accuracy_score(ys, y_pred):.4f} | "
          f"Precision: {precision_score(ys, y_pred):.4f} | "
          f"Recall: {recall_score(ys, y_pred):.4f} | "
          f"F1: {f1_score(ys, y_pred):.4f} | "
          f"ROC AUC: {roc_auc_score(ys, y_prob):.4f}")

val_pred = model_v2.predict(X_val_s)
val_prob = model_v2.predict_proba(X_val_s)[:, 1]

results_log.append({
    'Run': f'V2: L2, C={best_c}',
    'Val F1': f1_score(y_val, val_pred),
    'Val Precision': precision_score(y_val, val_pred),
    'Val Recall': recall_score(y_val, val_pred),
    'Val ROC AUC': roc_auc_score(y_val, val_prob),
})

---
## Run 5: Threshold Tuning

In [ ]:
y_prob_val = model_v2.predict_proba(X_val_s)[:, 1]

thresholds = np.arange(0.1, 0.91, 0.05)
t_results = []
for t in thresholds:
    yp = (y_prob_val >= t).astype(int)
    if yp.sum() == 0 or yp.sum() == len(yp):
        continue
    t_results.append({
        'Threshold': t,
        'Precision': precision_score(y_val, yp),
        'Recall': recall_score(y_val, yp),
        'F1': f1_score(y_val, yp),
    })

t_df = pd.DataFrame(t_results)
print(t_df.to_string(index=False))
best_t = t_df.loc[t_df['F1'].idxmax(), 'Threshold']
print(f"\nBest threshold: {best_t:.2f}")

In [ ]:
plt.figure(figsize=(10, 5))
plt.plot(t_df['Threshold'], t_df['Precision'], 'o-', label='Precision')
plt.plot(t_df['Threshold'], t_df['Recall'], 's-', label='Recall')
plt.plot(t_df['Threshold'], t_df['F1'], '^-', label='F1', linewidth=2)
plt.axvline(x=best_t, color='red', linestyle='--', alpha=0.5, label=f'Best={best_t:.2f}')
plt.axvline(x=0.5, color='gray', linestyle=':', alpha=0.5, label='Default (0.5)')
plt.xlabel('Threshold'); plt.ylabel('Score')
plt.title('Precision / Recall / F1 vs Threshold')
plt.legend(); plt.grid(True, alpha=0.3); plt.tight_layout()
plt.show()

In [ ]:
# Evaluate with best threshold on train + validation only
print(f"Using threshold = {best_t:.2f}\n")

for name, Xs, ys in [('Train', X_train_s, y_train), ('Validation', X_val_s, y_val)]:
    y_prob = model_v2.predict_proba(Xs)[:, 1]
    y_pred = (y_prob >= best_t).astype(int)
    print(f"[{name}] Accuracy: {accuracy_score(ys, y_pred):.4f} | "
          f"Precision: {precision_score(ys, y_pred):.4f} | "
          f"Recall: {recall_score(ys, y_pred):.4f} | "
          f"F1: {f1_score(ys, y_pred):.4f} | "
          f"ROC AUC: {roc_auc_score(ys, y_prob):.4f}")

val_prob = model_v2.predict_proba(X_val_s)[:, 1]
val_pred_t = (val_prob >= best_t).astype(int)

results_log.append({
    'Run': f'V3: L2, C={best_c}, t={best_t:.2f}',
    'Val F1': f1_score(y_val, val_pred_t),
    'Val Precision': precision_score(y_val, val_pred_t),
    'Val Recall': recall_score(y_val, val_pred_t),
    'Val ROC AUC': roc_auc_score(y_val, val_prob),
})

---
## Run 6: L1 Regularization (Lasso)

**Goal:** Compare L1 vs L2 regularization.

**Why L1?** L1 regularization (Lasso) drives some coefficients to exactly zero, effectively
performing feature selection during training. L2 (Ridge) shrinks all coefficients but keeps
them all non-zero. With 33 features where some may be irrelevant, L1 might outperform L2
by automatically zeroing out noise features.

**Solver change:** L-BFGS doesn't support L1, so we switch to SAGA (Stochastic Average Gradient
with variance reduction) which supports all regularization types.

In [ ]:
# Run 6: L1 Regularization
model_l1 = LogisticRegression(C=best_c, l1_ratio=1, solver='saga', max_iter=5000,
                               class_weight='balanced', random_state=42)
model_l1.fit(X_train_s, y_train)

print("L1 (Lasso) Regularization Results:")
for name, Xs, ys in [('Train', X_train_s, y_train), ('Validation', X_val_s, y_val)]:
    y_pred = model_l1.predict(Xs)
    y_prob = model_l1.predict_proba(Xs)[:, 1]
    print(f"[{name}] Accuracy: {accuracy_score(ys, y_pred):.4f} | "
          f"Precision: {precision_score(ys, y_pred):.4f} | "
          f"Recall: {recall_score(ys, y_pred):.4f} | "
          f"F1: {f1_score(ys, y_pred):.4f} | "
          f"ROC AUC: {roc_auc_score(ys, y_prob):.4f}")

# How many coefficients were zeroed out?
n_zero = (model_l1.coef_[0] == 0).sum()
zero_features = [f for f, c in zip(X_train_s.columns, model_l1.coef_[0]) if c == 0]
print(f"\nFeatures zeroed out by L1: {n_zero}/{len(X_train_s.columns)}")
if zero_features:
    print(f"Zeroed features: {zero_features}")

val_pred = model_l1.predict(X_val_s)
val_prob = model_l1.predict_proba(X_val_s)[:, 1]

results_log.append({
    'Run': f'V4: L1 (Lasso), C={best_c}',
    'Val F1': f1_score(y_val, val_pred),
    'Val Precision': precision_score(y_val, val_pred),
    'Val Recall': recall_score(y_val, val_pred),
    'Val ROC AUC': roc_auc_score(y_val, val_prob),
})

---
## Run 7: Elastic Net (L1 + L2 mix)

**Goal:** Try a blend of L1 and L2 regularization.

**Why Elastic Net?** It combines the strengths of both:
- L1's ability to zero out irrelevant features (sparsity)
- L2's stability when features are correlated (L1 can arbitrarily pick one of two correlated features and zero the other)

We use `l1_ratio=0.5` — equal mix of L1 and L2 penalties.

In [ ]:
# Run 7: Elastic Net
model_en = LogisticRegression(C=best_c, l1_ratio=0.5, solver='saga', max_iter=5000,
                               class_weight='balanced', random_state=42)
model_en.fit(X_train_s, y_train)

print("Elastic Net (l1_ratio=0.5) Results:")
for name, Xs, ys in [('Train', X_train_s, y_train), ('Validation', X_val_s, y_val)]:
    y_pred = model_en.predict(Xs)
    y_prob = model_en.predict_proba(Xs)[:, 1]
    print(f"[{name}] Accuracy: {accuracy_score(ys, y_pred):.4f} | "
          f"Precision: {precision_score(ys, y_pred):.4f} | "
          f"Recall: {recall_score(ys, y_pred):.4f} | "
          f"F1: {f1_score(ys, y_pred):.4f} | "
          f"ROC AUC: {roc_auc_score(ys, y_prob):.4f}")

n_zero = (model_en.coef_[0] == 0).sum()
print(f"\nFeatures zeroed out: {n_zero}/{len(X_train_s.columns)}")

val_pred = model_en.predict(X_val_s)
val_prob = model_en.predict_proba(X_val_s)[:, 1]

results_log.append({
    'Run': f'V5: Elastic Net (0.5), C={best_c}',
    'Val F1': f1_score(y_val, val_pred),
    'Val Precision': precision_score(y_val, val_pred),
    'Val Recall': recall_score(y_val, val_pred),
    'Val ROC AUC': roc_auc_score(y_val, val_prob),
})

In [ ]:
# Compare coefficients across L2, L1, and Elastic Net
coef_compare = pd.DataFrame({
    'Feature': X_train_s.columns,
    'L2 (Ridge)': model_v2.coef_[0],
    'L1 (Lasso)': model_l1.coef_[0],
    'Elastic Net': model_en.coef_[0],
})

fig, axes = plt.subplots(1, 3, figsize=(18, 8), sharey=True)

for ax, col, title in zip(axes, ['L2 (Ridge)', 'L1 (Lasso)', 'Elastic Net'],
                                  ['L2 (Ridge)', 'L1 (Lasso)', 'Elastic Net (0.5)']):
    sorted_df = coef_compare.sort_values(col)
    colors = ['#e74c3c' if c < 0 else ('#bdc3c7' if c == 0 else '#2ecc71') for c in sorted_df[col]]
    ax.barh(range(len(sorted_df)), sorted_df[col], color=colors)
    ax.set_yticks(range(len(sorted_df)))
    ax.set_yticklabels(sorted_df['Feature'], fontsize=8)
    ax.set_title(title)
    ax.axvline(x=0, color='black', linewidth=0.5)
    n_zero = (sorted_df[col] == 0).sum()
    ax.set_xlabel(f'Coefficient ({n_zero} zeroed)')

plt.suptitle('Coefficient Comparison: L2 vs L1 vs Elastic Net', fontsize=14, y=1.02)
plt.tight_layout()
plt.show()

---
## Run Summary (Validation Only)

All model selection is done based on validation metrics. The test set has NOT been touched yet.

In [ ]:
# Validation-only comparison
summary_df = pd.DataFrame(results_log)

print("=" * 90)
print("EXPERIMENT LOG — Model Selection (Validation Set Only)")
print("=" * 90)
print(summary_df.to_string(index=False))

# Pick best model by validation F1
best_idx = summary_df['Val F1'].idxmax()
print(f"\nBest run by Val F1: {summary_df.loc[best_idx, 'Run']}")
print(f"  Val F1:        {summary_df.loc[best_idx, 'Val F1']:.4f}")
print(f"  Val Precision: {summary_df.loc[best_idx, 'Val Precision']:.4f}")
print(f"  Val Recall:    {summary_df.loc[best_idx, 'Val Recall']:.4f}")
print(f"  Val ROC AUC:   {summary_df.loc[best_idx, 'Val ROC AUC']:.4f}")

# Store which model won
best_run_name = summary_df.loc[best_idx, 'Run']

---
## Cross-Notebook Comparison: All Features vs MRMR (Validation Only)

**Goal:** Compare the best model from this notebook (33 features) against the best model
from the MRMR notebook (20 features) using **validation metrics only** — no test set involved.

**Why this is valid:** Both pipelines use the same `random_state=42` and identical split ratios
(70/15/15 stratified), so the same rows land in train/validation/test. The only difference is
the feature set. Comparing on validation is fair and avoids test set bias.

**Why not compare on test?** If we evaluated every model variant on the test set and picked the
winner, the test score would be optimistically biased (adaptive overfitting). Model selection
must happen on validation; the test set is reserved for a single final evaluation of the chosen model.

In [ ]:
# --- Load & prepare MRMR dataset (20 features) with identical pipeline ---
df_mrmr = pd.read_csv('../data/processed/selected_features.csv')

y_mrmr = (df_mrmr['fraud_reported'] == 'Y').astype(int)
X_mrmr = df_mrmr.drop(columns=['fraud_reported'])

for col in X_mrmr.select_dtypes(include=['object', 'str']).columns:
    X_mrmr[col] = LabelEncoder().fit_transform(X_mrmr[col].astype(str))

# Same split logic & random_state — produces identical row assignments
X_temp_m, X_test_m, y_temp_m, y_test_m = train_test_split(
    X_mrmr, y_mrmr, test_size=0.15, random_state=42, stratify=y_mrmr)
X_train_m, X_val_m, y_train_m, y_val_m = train_test_split(
    X_temp_m, y_temp_m, test_size=0.15/(1-0.15), random_state=42, stratify=y_temp_m)

scaler_m = StandardScaler()
X_train_ms = pd.DataFrame(scaler_m.fit_transform(X_train_m), columns=X_train_m.columns, index=X_train_m.index)
X_val_ms = pd.DataFrame(scaler_m.transform(X_val_m), columns=X_val_m.columns, index=X_val_m.index)

print(f"MRMR dataset: {X_mrmr.shape[1]} features")
print(f"All-features dataset: {X.shape[1]} features")
print(f"Validation samples — MRMR: {len(X_val_m)} | All: {len(X_val)}")

# --- Train best MRMR model (same C search) ---
C_values_m = [0.001, 0.01, 0.1, 0.5, 1.0, 5.0, 10.0, 50.0, 100.0]
best_f1_m, best_c_m = 0, 1.0

for C in C_values_m:
    m = LogisticRegression(C=C, solver='lbfgs', max_iter=1000,
                           class_weight='balanced', random_state=42)
    m.fit(X_train_ms, y_train_m)
    vp = m.predict(X_val_ms)
    f1_val = f1_score(y_val_m, vp)
    if f1_val > best_f1_m:
        best_f1_m = f1_val
        best_c_m = C

model_mrmr = LogisticRegression(C=best_c_m, solver='lbfgs', max_iter=1000,
                                 class_weight='balanced', random_state=42)
model_mrmr.fit(X_train_ms, y_train_m)
print(f"\nMRMR best C: {best_c_m}")

# --- Threshold tuning for MRMR model ---
y_prob_val_m = model_mrmr.predict_proba(X_val_ms)[:, 1]
best_t_m, best_f1_t_m = 0.5, 0

for t in np.arange(0.1, 0.91, 0.05):
    yp = (y_prob_val_m >= t).astype(int)
    if yp.sum() == 0 or yp.sum() == len(yp):
        continue
    f1_t = f1_score(y_val_m, yp)
    if f1_t > best_f1_t_m:
        best_f1_t_m = f1_t
        best_t_m = t

print(f"MRMR best threshold: {best_t_m:.2f}")

In [ ]:
# --- Best configuration for All Features (from this notebook's runs) ---
# Use the best run's model + threshold already determined above
all_best_model = models[best_run_name]
all_use_threshold = best_t if 't=' in best_run_name else 0.5

y_prob_all_val = all_best_model.predict_proba(X_val_s)[:, 1]
y_pred_all_val = (y_prob_all_val >= all_use_threshold).astype(int)

# --- Best configuration for MRMR ---
y_prob_mrmr_val = model_mrmr.predict_proba(X_val_ms)[:, 1]
y_pred_mrmr_val = (y_prob_mrmr_val >= best_t_m).astype(int)

# --- Side-by-side comparison table ---
comparison = pd.DataFrame({
    'Metric': ['Features', 'Best C', 'Threshold', 'Val Accuracy', 'Val Precision',
               'Val Recall', 'Val F1', 'Val ROC AUC'],
    'All Features (33)': [
        X.shape[1], best_c, f'{all_use_threshold:.2f}',
        f"{accuracy_score(y_val, y_pred_all_val):.4f}",
        f"{precision_score(y_val, y_pred_all_val):.4f}",
        f"{recall_score(y_val, y_pred_all_val):.4f}",
        f"{f1_score(y_val, y_pred_all_val):.4f}",
        f"{roc_auc_score(y_val, y_prob_all_val):.4f}",
    ],
    'MRMR Selected (20)': [
        X_mrmr.shape[1], best_c_m, f'{best_t_m:.2f}',
        f"{accuracy_score(y_val_m, y_pred_mrmr_val):.4f}",
        f"{precision_score(y_val_m, y_pred_mrmr_val):.4f}",
        f"{recall_score(y_val_m, y_pred_mrmr_val):.4f}",
        f"{f1_score(y_val_m, y_pred_mrmr_val):.4f}",
        f"{roc_auc_score(y_val_m, y_prob_mrmr_val):.4f}",
    ],
})

print("=" * 70)
print("CROSS-NOTEBOOK COMPARISON (Validation Metrics Only)")
print("=" * 70)
print(comparison.to_string(index=False))

# --- Determine winner ---
f1_all = f1_score(y_val, y_pred_all_val)
f1_mrmr = f1_score(y_val_m, y_pred_mrmr_val)
winner = "All Features (33)" if f1_all > f1_mrmr else "MRMR Selected (20)"
print(f"\nWinner by Val F1: {winner} ({max(f1_all, f1_mrmr):.4f} vs {min(f1_all, f1_mrmr):.4f})")

In [ ]:
# --- Visual comparison: bar chart + ROC overlay ---
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Bar chart of key metrics
metrics = ['Precision', 'Recall', 'F1', 'ROC AUC']
all_vals = [
    precision_score(y_val, y_pred_all_val),
    recall_score(y_val, y_pred_all_val),
    f1_all,
    roc_auc_score(y_val, y_prob_all_val),
]
mrmr_vals = [
    precision_score(y_val_m, y_pred_mrmr_val),
    recall_score(y_val_m, y_pred_mrmr_val),
    f1_mrmr,
    roc_auc_score(y_val_m, y_prob_mrmr_val),
]

x = np.arange(len(metrics))
w = 0.35
axes[0].bar(x - w/2, all_vals, w, label=f'All Features (33)', color='#3498db')
axes[0].bar(x + w/2, mrmr_vals, w, label=f'MRMR (20)', color='#e67e22')
axes[0].set_xticks(x)
axes[0].set_xticklabels(metrics)
axes[0].set_ylabel('Score')
axes[0].set_title('Validation Metrics Comparison')
axes[0].legend()
axes[0].set_ylim(0, 1)
axes[0].grid(True, alpha=0.3, axis='y')

# Add value labels on bars
for i, (a, m) in enumerate(zip(all_vals, mrmr_vals)):
    axes[0].text(i - w/2, a + 0.02, f'{a:.3f}', ha='center', va='bottom', fontsize=9)
    axes[0].text(i + w/2, m + 0.02, f'{m:.3f}', ha='center', va='bottom', fontsize=9)

# ROC curves overlay
fpr_all, tpr_all, _ = roc_curve(y_val, y_prob_all_val)
fpr_mrmr, tpr_mrmr, _ = roc_curve(y_val_m, y_prob_mrmr_val)

axes[1].plot(fpr_all, tpr_all, 'b-', linewidth=2,
             label=f'All Features (AUC={auc(fpr_all, tpr_all):.3f})')
axes[1].plot(fpr_mrmr, tpr_mrmr, '-', color='#e67e22', linewidth=2,
             label=f'MRMR (AUC={auc(fpr_mrmr, tpr_mrmr):.3f})')
axes[1].plot([0, 1], [0, 1], 'k--', alpha=0.3, label='Random')
axes[1].set_xlabel('False Positive Rate')
axes[1].set_ylabel('True Positive Rate')
axes[1].set_title('ROC Curves — Validation Set')
axes[1].legend()
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print(f"\nConclusion: {winner} achieves a higher validation F1.")
print("This comparison uses only validation data — test set remains untouched for final evaluation.")

---
## Final Test Evaluation (ONE-TIME)

The best model was selected using only validation metrics above.
This is the **first and only** time the test set is evaluated — no decisions will be made after this.

In [ ]:
# Map best run to its model object
models = {
    'V1: L2, C=1.0': model_v1,
    f'V2: L2, C={best_c}': model_v2,
    f'V3: L2, C={best_c}, t={best_t:.2f}': model_v2,  # same model, different threshold
    f'V4: L1 (Lasso), C={best_c}': model_l1,
    f'V5: Elastic Net (0.5), C={best_c}': model_en,
}

best_model = models[best_run_name]
use_threshold = best_t if 't=' in best_run_name else 0.5

print(f"Selected model: {best_run_name}")
print(f"Threshold: {use_threshold}")
print()

# ONE-TIME test set evaluation
y_prob_test = best_model.predict_proba(X_test_s)[:, 1]
y_pred_test = (y_prob_test >= use_threshold).astype(int)

print("TEST SET RESULTS")
print("=" * 55)
print(f"Accuracy:  {accuracy_score(y_test, y_pred_test):.4f}")
print(f"Precision: {precision_score(y_test, y_pred_test):.4f}")
print(f"Recall:    {recall_score(y_test, y_pred_test):.4f}")
print(f"F1 Score:  {f1_score(y_test, y_pred_test):.4f}")
print(f"ROC AUC:   {roc_auc_score(y_test, y_prob_test):.4f}")
print()
print(classification_report(y_test, y_pred_test, target_names=['Non-Fraud', 'Fraud']))

In [ ]:
# Confusion matrix and ROC curve for final model
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Confusion matrix
cm = confusion_matrix(y_test, y_pred_test)
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', ax=axes[0],
            xticklabels=['Non-Fraud', 'Fraud'], yticklabels=['Non-Fraud', 'Fraud'])
axes[0].set_title(f'Confusion Matrix — {best_run_name}')
axes[0].set_ylabel('Actual'); axes[0].set_xlabel('Predicted')

# ROC Curve
fpr, tpr, _ = roc_curve(y_test, y_prob_test)
roc_auc_val = auc(fpr, tpr)
axes[1].plot(fpr, tpr, 'b-', linewidth=2, label=f'ROC (AUC = {roc_auc_val:.3f})')
axes[1].plot([0, 1], [0, 1], 'k--', alpha=0.3, label='Random')
axes[1].set_xlabel('False Positive Rate'); axes[1].set_ylabel('True Positive Rate')
axes[1].set_title('ROC Curve (Test Set)')
axes[1].legend(); axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

In [ ]:
# Feature coefficients of the best model
coef_df = pd.DataFrame({
    'Feature': X_train_s.columns,
    'Coefficient': best_model.coef_[0]
}).sort_values('Coefficient', ascending=True)

plt.figure(figsize=(10, 10))
colors = ['#e74c3c' if c < 0 else ('#bdc3c7' if c == 0 else '#2ecc71') for c in coef_df['Coefficient']]
plt.barh(range(len(coef_df)), coef_df['Coefficient'], color=colors)
plt.yticks(range(len(coef_df)), coef_df['Feature'])
plt.xlabel('Coefficient')
plt.title(f'Logistic Regression Coefficients — {best_run_name}\n(Green = fraud, Red = non-fraud, Gray = zeroed)')
plt.axvline(x=0, color='black', linewidth=0.5)
plt.tight_layout()
plt.show()